# Importing necessary modules from SageMath

In [1]:
from sage.all import *
import numpy as np
import math

# CP code encoder functions

In [2]:
approx = ComplexField(32)  # precision 


class GRSBase:
    """GRS subcode for CP code. 
    Note: This works for non-prime fields, unlike the CP code subclass. 
    Parameters:
        length - length of the code; this is equal to q - 1, where q is the 
            size of the base field 
        rs_dim - dimension of the associated (full) GRS code; this is equal 
            to k + 1, where k is the degree of the message polynomial
        chi_ind (optional, 1 by default) - index of the character function, 
            only used in the CP code subclass
    """
    def __init__(self, length, rs_dim, chi_ind=1):
        # parameters
        self.length = length
        self.rs_dim = rs_dim
        # finite field
        self.field_size = length + 1
        F_q.<w> = GF(self.field_size, modulus="primitive")
        self.field = F_q  # w = self.field.gen()
        self.field_char = self.field.characteristic()
        self.field_units = [w^i for i in range(length)]
        # alternatively,
        #self.field = GF(self.field_size)
        #self.field_units = self.field.list()[1:self.field_size]
        # check that we got all the units 
        #assert(len(self.field_units) == self.length and 
        #       self.field_units == set(self.field.list()[1:self.field_size]))
        self.chi_ind = chi_ind % self.field_char
        # polynomial ring
        F_q_x.<x> = self.field[]  # F_q[x]
        self.poly_ring = F_q_x  # x = self.polynomial_ring().gen()
        # GRS
        self.grs = codes.GeneralizedReedSolomonCode(
            self.field_units, 
            self.rs_dim, 
            self.field_units
        )
        self.grs_decoder = codes.decoders.GRSBerlekampWelchDecoder
        self.grs_list_decoder = codes.decoders.GRSGuruswamiSudanDecoder
        # pre-compute dimension and minimum distance for __str__ and __eq__
        self._dim = self.dimension()
        self._dmin = self.minimum_distance()
    
    def __str__(self):
        return (f"({self.length}, {self._dim}, {self._dmin}) CP code with "
                f"character chi_{self.chi_ind} associated with {self.grs}")
    
    def __repr__(self):
        return self.__str__()
    
    def __eq__(self, other):
        if isinstance(other, GRSBase):
            return (self._dim == other._dim and self._dmin == other._dmin)
        return NotImplemented
    
    def polynomial_ring(self):
        """Return the polynomial ring of self."""
        return self.poly_ring
    
    def __grs_generator_matrix(self):
        """Return a generator matrix of the GRS subcode of self."""
        mat = copy(self.grs.generator_matrix())
        #print(mat)
        #print()
        p = self.field_char
        for i in range(self.rs_dim):
            if i % p == 0:
                mat[i,:] = 0
        #print(mat)
        return mat
    
    def grs_subcode(self): 
        """Return the GRS subcode of self.grs."""
        return LinearCode(self.__grs_generator_matrix())
    
    def dimension(self):
        return self.grs_subcode().dimension()
    
    def minimum_distance(self):
        return self.grs_subcode().minimum_distance()
    
    def covering_radius(self):
        return self.grs_subcode().covering_radius()


class CharacterPolynomialCode(GRSBase):
    def __init__(self, length, rs_dim, chi_ind=1):
        if not is_prime(length + 1):
            raise ValueError("This implementation only works for prime fields")
        super().__init__(length, rs_dim, chi_ind)
        # pre-compute character function and its inverse
        self.__chi_mem = dict()
        self.__phi_mem = dict()
        for a in self.field:
            self.__chi_mem[a] = self.__chi(a)
            self.__phi_mem[self.__chi_mem[a]] = a
        #print(f"chi: (size {len(self.__chi_mem)})\n{self.__chi_mem}")
        #print(f"phi: (size {len(self.__phi_mem)})\n{self.__phi_mem}")

    def __contains__(self, c):
        """
        If c is a polynomial, return True iff c is in the message space of self. 
        If c is a vector, return True iff c is a valid codeword of self. 
        """
        if isinstance(
            c, 
            sage.rings.polynomial.polynomial_zmod_flint.Polynomial_zmod_flint
        ):
            for i, coeff in enumerate(c.list()):  # iterate over coeffs of c
                if i % self.field_char == 0 and coeff != 0:
                    return False
            return c in self.poly_ring and c.degree() < self.rs_dim
        if isinstance(
            c, 
            sage.modules.free_module_element.FreeModuleElement_generic_dense
        ):
            try:
                f = self.decode(c) 
                return self.encode(f) == c
            except sage.coding.decoder.DecodingError:
                return False
        return False
    
    def dimension(self):
        k = self.rs_dim
        return k - k//self.field_char
    
    def minimum_distance(self):
        return self.field_size - self.rs_dim
    
    def covering_radius(self):
        return self.minimum_distance() - 1
    
    def decoding_radius(self):
        return self.grs_decoder(self.grs).decoding_radius()
    
    def list_decoding_radius(self):
        return self.grs_list_decoder.guruswami_sudan_decoding_radius(self.grs)[0]
    
    def convert_polynomial(self, f):
        """Convert f to a polynomial in the message space of self by setting 
        every p-th coefficient equal to 0, where p = self.field_char.
        """
        if not (f in self.poly_ring and f.degree() < self.rs_dim):
            raise ValueError(f"{f} is not an element of {self.poly_ring} "
                             f"of degree less than {self.rs_dim}")
        coeffs = f.list()
        for i in range(len(coeffs)):
            if i % self.field_char == 0:
                coeffs[i] = 0
        return self.poly_ring(coeffs)
    
    def __chi(self, a):
        """Compute the additive character of self.field corresponding to a. 
        This is completely determined by its value at 1, which can be any 
        p-th root of unity different from 1, where p = self.field_char.[1] 
        See also [2, Theorem 2.9]. 
        References:
        [1] https://www.imsc.res.in/~amri/html_notes/notesap2.html
        [2] https://people.math.rochester.edu/faculty/iosevich/ffwaring.pdf
        """
        if not a in self.field:
            raise ValueError(f"{a} is not an element of {self.field}")
        ja = self.chi_ind * a
        return exp(2*pi*I*lift(ja.trace())/self.field_char)
    
    def __encode_raw(self, f):
        """Encode f directly, i.e. without using the built-in GRS encoder."""
        cp_rs_cw = vector(
            self.field, 
            [a * f(a) for a in self.field_units]
        )
        cp_vec = [self.__chi_mem[c] for c in cp_rs_cw]
        return vector(approx, cp_vec)
    
    def encode(self, f, debug=False):
        if not f in self:
            raise ValueError(f"{f} is not an element of the message space "
                             f"of {self}")
        g = f // self.poly_ring.gen()
        cp_rs_cw = self.grs.encoder("EvaluationPolynomial").encode(g)
        if debug:
            print(cp_rs_cw)
        cp_vec = [self.__chi_mem[c] for c in cp_rs_cw]
        cp_cw = vector(approx, cp_vec)
        if debug:
            print("Encoder success?", cp_cw == self.__encode_raw(g))
        return cp_cw
    
    def __nearest_img(self, z):
        """Return the nearest point (in Euclidean distance) to z in chi(F_q)."""
        p = self.field_char
        z = approx(z)
        
        if z.arg() >= 0:  # note: -pi < z.arg() <= pi
            ex = floor(self.length*approx(z.arg())/(2*pi) + 1/2)
        else: 
            # since -pi < arg(z) <= pi (see https://is.gd/M4aWuP), we first add
            # pi to arg(z), then shift the exponent by (p+1)//2
            ex = (p+1)//2 + floor(self.length*approx(pi+z.arg())/(2*pi) + 1/2)

        return exp(2*pi*I/p)^ex
    
    def __phi(self, z):
        """Return the pre-image of the nearest point to z in chi(F_q). 
        Note: This is not unique for an arbitrary q! In particular, for any b 
        in F_p, there are exactly q^(m-1) elements c in F_q with c.trace() = b, 
        where q = p^m.[1] 
        References:
        [1] https://en.wikipedia.org/wiki/Field_trace
        """
        z_to_unit_circle = self.__nearest_img(z)
        return self.__phi_mem.get(z_to_unit_circle)
    
    def __pre_process(self, m, debug=False):
        """Return the result of applying phi to each coordinate of m."""
        assert(len(m) == self.length)
        y = [0] * self.length
        for i, mi in enumerate(m):
            y[i] = self.__phi(mi)
        ynew = vector(self.field, y)
        if debug:
            print(f"After phi:\n\t{ynew}")
        return ynew

    def __pre_process_psk(self, w_opt, debug=False):
        """Return the result of applying phi to each coordinate of m."""
        
        assert(len(w_opt) == self.length)
        y = [0] * self.length
        # p = self.field_char

        # print((exp(pi*I/p)))
        
        eta = math.cos( (2*math.pi)/self.field_size ) + 1j * math.sin( (2*math.pi)/self.field_size )
        # print((exp(pi*I/p)))
        
        arg = (np.angle(w_opt) * self.field_size) / ( 2*math.pi )
        g = round(arg)
        u = np.argsort(g - arg)
        eta_gt = np.array([])
        
        for i in range(g.shape[0]):
            eta_gt = np.append (eta_gt, math.cos( (2*math.pi * g[i])/self.field_size ) + 1j * math.sin( (2*math.pi * g[i] )/self.field_size ))
        p = w_opt * eta_gt
        v = np.hstack((np.sum(p), p[u]*(eta-1)))
        
        best = np.argmax(np.abs(np.cumsum(v)))
        g[u[1:best-1]] = g[u[1:best-1]] + 1

        g = (g-g[0])%(self.field_size)

        
        for i in range(w_opt.shape[0]):
            index = Integer(g[i])
            # print(index)
            y[i] = self.__phi_mem.get(exp(2*pi*I/self.field_size)^index)
            # print(y[i], exp(2*pi*I/self.field_size)^index)
        ynew = vector(self.field, y)
        return ynew
        
    
    def decode(self, m, debug=False):
        if debug:
            print(f"Running CP decoder using {self.grs_decoder} on received "
                  f"message {m}")
        ynew = self.__pre_process(m, debug)
        
        return (self.poly_ring.gen() * self.grs_decoder(self.grs).decode_to_message(ynew))
    
    def list_decode(self, m, debug=False):
        if debug:
            print(f"Running CP list decoder using {self.grs_list_decoder} on "
                  f"received message {m}")
        
        # ynew = self.__pre_process(m, debug)
        
        ynew = self.__pre_process_psk(m, debug)
        # print(ynew)
        gs_output = self.grs_list_decoder(self.grs, tau=self.list_decoding_radius()
        ).decode_to_message(ynew)
        grs_list = [self.poly_ring.gen() * f for f in gs_output]
        # print(grs_list)
        return [f for f in grs_list if f in self]

# Define GF parameters 

In [3]:
import numpy as np
import cmath

## Data transmission ##

# Length of the CP code. Must be a prime NUMBER-1.
length = 6

# GRS dimension "k". Degree of the GRS polynomial is dimension - 1. 
dimension = 1

# Character index 
chi_i = 1 

# Decoding radius ! = 0
# assert dimension < length 

# CP code condition
assert chi_i <= (length)

code = CharacterPolynomialCode(length, dimension, chi_i)

# code.decoding_radius()
# code.list_decoding_radius()

# Generate all CP codewords over GF(p)

In [4]:
F = GF(length+1)

R.<x> = PolynomialRing(F)

degree = dimension-1

# polynomials = (list(R.polynomials(max_degree=degree)))
polynomials = R.polynomials(max_degree=degree)


codebook = []
poly_c = set()

for poly in polynomials:
    poly_c.add(code.convert_polynomial(poly))
    
for poly in poly_c:
    codebook.append(code.encode(poly))
    
codebook = np.array(codebook)
# print(codebook.shape)

# Generate Rayleigh channel co-efficients

In [5]:
# Nt = length # Transmit antennas
# Nr = 1 # Recieve antenna
# # Nr = p_1 # Recieve antennas


# # ---- Rayleigh flat channel ---- #

# # H_mu, H_var = 0, math.sqrt(4/math.pi - 1)
# H_mu, H_var = 0, math.sqrt(1/2)
# # H_mu, H_var = 0, 1


# H_T = np.empty((0, Nt))

# np.random.seed(42)

# ##### Monte Carlo Simulation ####

# N_sim = 300

# for i in range(N_sim):
    
#     ht = ((np.random.normal(H_mu, H_var, (Nr, Nt))) + 1j * (np.random.normal(H_mu, H_var, (Nr ,Nt))))
#     H_T = np.vstack((H_T, ht))

    

# Generate Rician channel co-efficients

In [6]:
# Nt = length # Transmit antennas
# Nr = 1 # Recieve antenna

# # Nr = p_1 # Recieve antennas

# # ---- Rician flat fading channel ---- #

# # H_mu, H_var = 0, 1/2
# # H_mu, H_var = 0, 1
# H_mu, H_var = 0, sqrt(1/2)
# H_T = np.empty((0, Nt))

# np.random.seed(42)


# ##### Monte Carlo Simulation ####

# N_sim = 300

# for i in range(N_sim):
    
#     K_factor = 0.25
#     los = math.sqrt(K_factor/(K_factor+1))
#     nlos = math.sqrt(1/(K_factor+1)) * ((np.random.normal(H_mu, H_var, (Nr, Nt))) + 1j * (np.random.normal(H_mu, H_var, (Nr ,Nt))))

#     ht = los + nlos
    
#     H_T = np.vstack((H_T, ht))

# Noisy Channel Measurements

In [7]:
Nt = length # Transmit antennas
Nr = 1 # Recieve antenna
# Nr = p_1 # Recieve antennas


# ---- Rayleigh flat channel ---- #

# H_mu, H_var = 0, math.sqrt(4/math.pi - 1)
H_mu, H_std = 0, math.sqrt(1/2)

# H_mu, H_var = 0, 1


H_T = np.empty((0, Nt), dtype=np.float64)

H_T_error = np.empty((0, Nt), dtype=np.float64)
error_std = math.sqrt(0.005)

np.random.seed(42)

##### Monte Carlo Simulation ####

N_sim = 300

for i in range(N_sim):
    
    ht = ((np.random.normal(H_mu, H_std, (Nr, Nt))) + 1j * (np.random.normal(H_mu, H_std, (Nr ,Nt))))
    H_T = np.vstack((H_T, ht))

    ht = ht + ((np.random.normal(H_mu, error_std, (Nr, Nt))) + 1j * (np.random.normal(H_mu, error_std, (Nr ,Nt))))
    H_T_error = np.vstack((H_T_error, ht))
    

# Find the optimal cp_cw by linear search

In [8]:
def decode(x, M):
    eta = math.cos( (2*math.pi)/M ) + 1j * math.sin( (2*math.pi)/M )
    arg = (np.angle(x) * M) / ( 2*math.pi )
    g = round(arg)
    u = np.argsort(g - arg)
    eta_gt = np.array([])
    for i in range(g.shape[0]):
        eta_gt = np.append (eta_gt, math.cos( (2*math.pi * g[i])/M ) + 1j * math.sin( (2*math.pi * g[i] )/M ))
    p = w_opt * eta_gt
    v = np.hstack((np.sum(p), p[u]*(eta-1)))
    
    best = np.argmax(np.abs(np.cumsum(v)))
    g[u[1:best-1]] = g[u[1:best-1]] + 1
    
    g = (g-g[0])%M

    return g

In [9]:
q_error = np.array([]) 
MRC_gain = np.array([])
CP_gain = np.array([])
EGT_gain = np.array([])
H_norm1 = np.array([])
cos2_max = 0
Sigma_tot = 0


for i in range(N_sim): 

    # H = H_T[i]
    
    # H = H_T_error[i]
    # H = H.reshape(1,-1)

    H_true = H_T[i]
    H_true = H_true.reshape(1,-1)

    
    H_estimate = np.array((H_std * (1/(H_std + error_std))) * H_true)

    # print(H_true.shape, H_estimate.shape, type(H_true), type(H_estimate), H_true, H_estimate)
    # print(H)

    # ---- MIMO channel 

    U, Sigma, Vh = np.linalg.svd(H_true, full_matrices=False)
    Sigma_tot += Sigma

    # Find the ideal infinite precision vector
    
    if Nr == 1:
        w_opt = np.conjugate(np.transpose(Vh))
        w_opt = w_opt.reshape(-1)
    else:
        w_opt = np.conjugate(np.transpose(Vh[0,:]))
        w_opt = w_opt.reshape(-1)

    # print(w_opt)
    # Find the maximum cos^2_theta value with linear search

    # beta has no effect as it's a scalar from lower field
    # w_opt = -1j* w_opt
    
    cos2_theta = np.zeros(codebook.shape[0])

    for cw in range(codebook.shape[0]):
        codebook[cw]/= np.linalg.norm(codebook[cw]) # TX unit power requirement
        cos2_theta[cw] = (np.abs(np.vdot(codebook[cw], w_opt))**2)
    
    theta_max_index = np.argmax(cos2_theta)
    cos2_max += np.max(cos2_theta) 

    v = codebook[theta_max_index]
    v = codebook[theta_max_index]/np.linalg.norm(codebook[theta_max_index])
    
    #  Receiver gains
    
    MRC_gain = np.append(MRC_gain, (np.linalg.norm(H_true @ w_opt))**2)
     
    CP_gain = np.append(CP_gain, (np.linalg.norm(H_true @ v ))**2)

    if Nr == 1:
            
        EGT_gain = np.append(EGT_gain, ((np.sum(np.abs(H_true)))**2)/Nt)
        
    else:
        for m in range(Nr):
            H_norm1 = np.append(H_norm1, (np.sum(np.abs(H[m, :])))) 

        EGT_gain = np.append(EGT_gain, ((np.max(H_norm1)**2)/Nt))
        
    q_error = np.append(q_error, np.linalg.norm(w_opt - v, 2)**2)

MRC_gain_tot = (np.sum(MRC_gain))/N_sim
EGT_gain_tot = (np.sum(EGT_gain))/N_sim
CP_gain_tot = (np.sum(CP_gain))/N_sim


MRC_gain_tot_db = 10*math.log10(MRC_gain_tot)
EGT_gain_tot_db = 10*math.log10(EGT_gain_tot)
CP_gain_tot_db = 10*math.log10(CP_gain_tot)


print("Receiver gains:\n",MRC_gain_tot, EGT_gain_tot, CP_gain_tot) 
print("\nGains in db:\n", MRC_gain_tot_db, EGT_gain_tot_db, CP_gain_tot_db)
print("\nCos2_max:\n", cos2_max/N_sim)
print("\nQuantization error:\n", (np.sum(q_error))/N_sim)
# print(Sigma, Sigma@Vh, np.linalg.norm(H), U)

Receiver gains:
 6.166216745201618 5.066336632251708 0.9439512459309032

Gains in db:
 7.900187862988004 7.046940430684651 -0.25050435966726714

Cos2_max:
 0.15587845704286085

Quantization error:
 2.1498578352203057


In [10]:
M = 16 # Modulation order
PSK_gain = np.array([])

for i in range(N_sim): 
    
    H_true = H_T[i]
    H_true = H_true.reshape(1,-1)

    H_estimate = np.array((H_std * (1/(H_std + error_std))) * H_true)

    # ---- MIMO channel 

    U, Sigma, Vh = np.linalg.svd(H_estimate, full_matrices=False)
    Sigma_tot += Sigma

    # Find the ideal infinite precision vector
    
    if Nr == 1:
        w_opt = np.conjugate(np.transpose(Vh))
        w_opt = w_opt.reshape(-1)
    else:
        w_opt = np.conjugate(np.transpose(Vh[0,:]))
        w_opt = w_opt.reshape(-1)


    psk_vec = np.array([])

    psk_ind = decode(w_opt, M)
    
    for k in range(w_opt.shape[0]):
        psk_vec = np.append(psk_vec, ( math.cos((2*math.pi/M)*psk_ind[k]) + 1j*math.sin((2*math.pi/M)*psk_ind[k]) ) / math.sqrt(length) )
    
    PSK_gain = np.append(PSK_gain, (np.linalg.norm(H_estimate @ psk_vec ))**2)

    q_error_psk = np.append(q_error, np.linalg.norm(w_opt - psk_vec, 2)**2)

PSK_gain_tot = (np.sum(PSK_gain))/ N_sim
PSK_gain_tot_db = 10*math.log10(PSK_gain_tot)

print("PSK gain:\n", PSK_gain_tot)
print("\nPSK Gain in db:\n", PSK_gain_tot_db)
print("\nPSK Quantization error:\n", (np.sum(q_error_psk))/N_sim)

PSK gain:
 4.085330491631461

PSK Gain in db:
 6.112271954794721

PSK Quantization error:
 2.1615234303866937


# List decoder

In [11]:
# q_error = np.array([]) 
# MRC_gain = np.array([])
# CP_gain = np.array([])
# EGT_gain = np.array([])
# H_norm1 = np.array([])
# cos2_max = 0
# Sigma_tot = 0
# count = 0


# for i in range(N_sim): 

#     H = H_T[i]
#     H = H.reshape(1,-1)
#     # print(H)

#     # ---- MIMO channel 

#     U, Sigma, Vh = np.linalg.svd(H, full_matrices=False)
#     Sigma_tot += Sigma

#     # Find the ideal infinite precision vector
    
#     if Nr == 1:
#         w_opt = np.conjugate(np.transpose(Vh))
#         w_opt = w_opt.reshape(-1)
#     else:
#         w_opt = np.conjugate(np.transpose(Vh[0,:]))
#         w_opt = w_opt.reshape(-1)

#     # print(w_opt)
#     # Find the maximum cos^2_theta value with linear search

#     # beta has no effect as it's a scalar from lower field
#     # w_opt = -1j* w_opt
    
#     # cos2_theta = np.zeros(codebook.shape[0])

#     # for cw in range(codebook.shape[0]):
#     #     codebook[cw]/= np.linalg.norm(codebook[cw]) # TX unit power requirement
#     #     cos2_theta[cw] = (np.abs(np.vdot(codebook[cw], w_opt))**2)
    
#     # theta_max_index = np.argmax(cos2_theta)
#     # cos2_max += np.max(cos2_theta) 

#     # v = codebook[theta_max_index]
#     # v = codebook[theta_max_index]/np.linalg.norm(codebook[theta_max_index])

#     # Find a beta that maps to nearest possible decoding_radius()
    
#     v_dec = code.list_decode(w_opt)

    
#     if v_dec:

#         count+=1
        
#         v = np.array(code.encode(v_dec[0])) # v_dec contains the decoded messgae polynomial. Need to find the corresponding cp_cw
#         # print(v)
#         v = v/np.linalg.norm(v) # Normalize the cp_cw

#         CP_gain = np.append(CP_gain, (np.linalg.norm(H @ v ))**2)
    
#     #  Receiver gains
    
#     MRC_gain = np.append(MRC_gain, (np.linalg.norm(H @ w_opt))**2)
     
    

#     if Nr == 1:
            
#         EGT_gain = np.append(EGT_gain, ((np.sum(np.abs(H)))**2)/Nt)
        
#     else:
#         for m in range(Nr):
#             H_norm1 = np.append(H_norm1, (np.sum(np.abs(H[m, :])))) 

#         EGT_gain = np.append(EGT_gain, ((np.max(H_norm1)**2)/Nt))
        
#     # q_error = np.append(q_error, np.linalg.norm(w_opt - v, 2)**2)

# print(count)

# MRC_gain_tot = (np.sum(MRC_gain))/N_sim
# EGT_gain_tot = (np.sum(EGT_gain))/N_sim
# CP_gain_tot = (np.sum(CP_gain))/count


# MRC_gain_tot_db = 10*math.log10(MRC_gain_tot)
# EGT_gain_tot_db = 10*math.log10(EGT_gain_tot)
# CP_gain_tot_db = 10*math.log10(CP_gain_tot)


# print("Receiver gains:\n",MRC_gain_tot, EGT_gain_tot, CP_gain_tot)
# print("\nGains in db:\n", MRC_gain_tot_db, EGT_gain_tot_db, CP_gain_tot_db)
# print("\nCos2_max:\n", cos2_max/N_sim)
# print("\nQuantization error:\n", (np.sum(q_error))/N_sim)


# # print(Sigma, Sigma@Vh, np.linalg.norm(H), U)

# Theoretical bound comparision

In [12]:
# # Utmost loss in db

# import numpy as np
# k_least = 5
# n = 6

# cos2_theta = np.array([])

# # print(len(range(k_least, n+1)))

# for k in range(k_least, n+1):
#     # print(k)
#     bound = (2*k)/n - 1 - math.sqrt(4/pi - 1)
#     # print(bound)
#     cos2_theta = np.append(cos2_theta, bound**2)
#     print(10*math.log10(cos2_theta[k-k_least]))
# print(cos2_theta)

# MRC and EGT gain difference

In [13]:
# q_error = np.array([]) 
# MRC_gain = np.array([])
# CP_gain = np.array([])
# EGT_gain = np.array([])
# H_norm1 = np.array([])
# cos2_max = 0
# Sigma_tot = 0
# for i in range(N_sim): 

#     H = H_T[i]
#     H = H.reshape(1,-1)

#     # ---- MIMO channel 
    
#     U, Sigma, Vh = np.linalg.svd(H, full_matrices=False)

#     EGT_gain = np.append(EGT_gain, ((np.sum(np.abs(H)))**2)/Nt)

#     CP_gain = np.append(CP_gain, (Sigma**2) * cos2_theta[2])



# EGT_gain_tot = (np.sum(EGT_gain))/N_sim
# CP_gain_tot = (np.sum(CP_gain))/N_sim

# EGT_gain_tot_db = 10*math.log10(EGT_gain_tot)
# CP_gain_tot_db = 10*math.log10(CP_gain_tot)

# print("Receiver gains:\n", EGT_gain_tot, CP_gain_tot)
# print("\nGains in db:\n",  EGT_gain_tot_db, CP_gain_tot_db)    

# Find the beamforming vector using closest in_angle

In [14]:
# # beta = np.array([1 + 1j*0, 0.5 + 1j*0.866, -0.5 + 1j*0.866, -1 + 1j* 0, -0.5 - 1j*0.866, 0.5 -1j*0.866, 1 - 1j*0])
# cp_sum = 0
# for cw in range(codebook.shape[0]):
#     cp_sum += np.linalg.norm(codebook[cw])

# beta = (math.sqrt((codebook.shape[0]/cp_sum)))
# beta

In [15]:
# # print(w_opt[1]*beta[4], w_opt[1] , np.angle(w_opt[1]*beta[4]), np.angle(w_opt[1]), np.angle(w_opt))
# print(codebook[10], np.vdot(codebook[10], w_opt) )

# Find the beamforming vector using decoder

In [16]:
# q_error = np.array([]) 
# cos_in_angle = np.array([])
# MRC_gain = np.array([])
# CP_gain = np.array([])
# EGT_gain = np.array([])
# H_norm1 = np.array([])    
# count = 0

# for i in range(N_sim): 
    
#     H = H_T[i]
#     H = H.reshape(1,-1)

#     # ---- MIMO channel 
    
#     U, Sigma, Vh = np.linalg.svd(H, full_matrices=False)

#     if Nr == 1:
#         w_opt = np.conjugate(np.transpose(Vh))
#         w_opt = w_opt.reshape(-1)
#     else:
#         w_opt = np.conjugate(np.transpose(Vh[0,:]))
#         w_opt = w_opt.reshape(-1)
    
#     #  Receiver gains
    
#     MRC_gain = np.append(MRC_gain, (np.linalg.norm(H @ w_opt))**2)

#     if Nr == 1:
            
#         EGT_gain = np.append(EGT_gain, ((np.sum(np.abs(H)))**2)/Nt)
        
#     else:
#         for m in range(Nr):
#             H_norm1 = np.append(H_norm1, (np.sum(np.abs(H[m, :])))) 

#         EGT_gain = np.append(EGT_gain, ((np.max(H_norm1)**2)/Nt))        
    
#     # Find a beta that maps to nearest possible decoding_radius()
    
#     v_dec = code.list_decode(w_opt)
    
#     if v_dec:

#         count+=1
        
#         v = np.array(code.encode(v_dec[0])) # v_dec contains the decoded messgae polynomial. Need to find the corresponding cp_cw
#         v = v/np.linalg.norm(v) # Normalize the cp_cw
     
#         # # Find the cosine angle b/w w_opt and cp_cw
        
#         cos_in_angle = np.append(cos_in_angle, ((np.abs(np.vdot(w_opt, v)))))
#         # print(cos_in_angle)

#         CP_gain = np.append(CP_gain, (np.linalg.norm(H @ v))**2)

            
#         q_error = np.append(q_error, np.linalg.norm(w_opt-v, 2)**2)

# MRC_gain_tot = (np.sum(MRC_gain))/N_sim
# EGT_gain_tot = (np.sum(EGT_gain))/N_sim
# CP_gain_tot = (np.sum(CP_gain))/count

# MRC_gain_tot_db = 10*math.log10(MRC_gain_tot)
# EGT_gain_tot_db = 10*math.log10(EGT_gain_tot)
# CP_gain_tot_db = 10*math.log10(CP_gain_tot)

# # print("Receiver gains:\n",MRC_gain_tot, EGT_gain_tot, CP_gain_tot)
# print("\nGains in db:\n", MRC_gain_tot_db, EGT_gain_tot_db, CP_gain_tot_db)
# print("\nDecoder success count:\n",count)
# print("\nCos^2(angle):\n", (np.sum(cos_in_angle)/count)**2)
# print("\nQuantization error:\n", (np.sum(q_error))/count)


# # print("\nSigma:\n", Sigma, Sigma**2)

# SVD verification

In [17]:
# np.random.seed(42)
# H = ((np.random.normal(H_mu, H_var, (Nr, Nt))) + 1j * (np.random.normal(H_mu, H_var, (Nr ,Nt))))
# H = H*(1/math.sqrt(2))

# # print(H)
# # ---- MIMO channel 
    
# U, Sigma, Vh = np.linalg.svd(H, full_matrices=False)


# # print(Vh.shape)
# # Sigma = Sigma.reshape(1,-1)

# # Sigma2 = np.zeros((1, 12))
# # Sigma2[0,0] = Sigma[0]
# # print(Sigma2)
# # print(U.shape, Sigma.shape, Vh.shape)

# # print((U @ Sigma2)@(Vh))
# test_opt = np.conjugate(np.transpose(Vh))

# print(np.linalg.norm(H @ test_opt), Sigma)

# Plot the results

In [18]:
# print(f"Decoder % :{100*float(count/N_sim)}", f"count :{float(count)}") 
# print(f"MRC_gain :{MRC_gain_db}", f"EGT_gain :{EGT_gain_db}", f"CP_gain :{CP_gain_db}") 
# print(f"q_error :{q_error}") 
# print(cos_in_angle)
   
# print(MRC_gain)

# x = np.linspace(1, count, count)
# plt.plot(x, MRC_gain_db, linewidth = 0.5, linestyle='-', marker='v', color='g', label="MRT")
# plt.plot(x, EGT_gain_db, linewidth = 0.5, linestyle='-', marker='o', color='b', label="EGT")
# plt.plot(x, CP_gain_db, linewidth = 0.5, linestyle='-', marker='*', color='c',  label="CP")
# plt.xlabel("CP decoder success")
# plt.ylabel("Average gain")
# plt.legend()
# plt.show()

# Block code decoder issue

In [19]:
# # Generate random message co-efficients #

# import numpy as np

# N_msg = 1
# rows, cols, low, high = N_msg , (k), np.min(code.field), np.max(code.field) 
# message_coeff = np.random.randint(low, high, (rows, cols))

# # message_coeff = np.array([[3, 0, 1, 0], [4, 0, 0, 0]])

# x = code.polynomial_ring().gen()
# cp_cws = np.empty((0,p_1), dtype=complex)
# message_coeff_d = np.empty((0,k), dtype=complex)

# for a in message_coeff:
    
#     poly = code.poly_ring(list(a))
#     g = code.convert_polynomial(poly)
#     cp_cw = np.array(code.encode(g))  # cp codeword
#     cp_cws = np.vstack((cp_cws, cp_cw))
#     # assert code.decode(cp_cw) == g
#     # print(code.decode(cp_cw), g)
#     # print(i, cp_cw, list(a), message_coeff[i,:], np.array(code.decode(cp_cw).list()))
#     # message_coeff_d = np.vstack((message_coeff_d, np.array(code.decode(cp_cw).list())))
#     # i += 1
#     print(np.array(code.list_decode(cp_cw)))
          
# # TX and RX are agreed on mod p co-efficients

# # message_coeff_d[:, 0] = message_coeff[:, 0] 
# # assert np.array_equal(message_coeff_d, message_coeff)

